In [1]:
pip install rugbypy

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [36]:
from rugbypy.match import *
from rugbypy.competition import *
from rugbypy.team import *
from rugbypy.venue import *
from rugbypy.player import *

In [37]:
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

In [46]:
## Reading in data

test_matches = pd.read_excel("detailed_test_matches.xlsx")
match_stats = pd.read_excel("aggregated_stats.xlsx")
test_nations = pd.read_excel("test_nations.xlsx")
team_stats = pd.read_excel("team_stats.xlsx")
venues = fetch_all_venues()

Fetching all venues...


In [47]:
team_stats

,team_id,team,game_date,team_vs,team_vs_id,match_id,22m_entries,22m_conversion,line_breaks,carries,...,offload,pc_possession_first,pc_possession_second,pc_territory_first,pc_territory_second,rucks_total,runs,scrums_total,total_free_kicks_conceded,turnover_knock_on
0,67d88578,France,20220206,Italy,d0cffe5e,4a49c716,NaN,NaN,7,NaN,...,9.0,0.62,0.54,0.66,0.64,96.0,124.0,5.0,0.0,7.0
1,67d88578,France,20220212,Ireland,165b36e5,71ccf2fb,NaN,NaN,3,NaN,...,8.0,0.52,0.43,0.55,0.48,83.0,101.0,5.0,1.0,7.0
2,67d88578,France,20220226,Scotland,4409215c,eb863c39,NaN,NaN,6,NaN,...,10.0,0.45,0.39,0.39,0.61,41.0,75.0,9.0,1.0,6.0
3,67d88578,France,20220311,Wales,094dae60,6b6585fc,NaN,NaN,4,NaN,...,10.0,0.53,0.41,0.55,0.33,69.0,98.0,6.0,1.0,9.0
4,67d88578,France,20220319,England,f60f757d,cc3c0a71,NaN,NaN,3,NaN,...,13.0,0.51,0.39,0.60,0.27,71.0,98.0,9.0,0.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524,3a1983b3,Fiji,20241123,Ireland,165b36e5,8923cd55,17.0,3.06,13,136.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
525,3a1983b3,Fiji,20250830,Tonga,be110f80,b901c3ad,12.0,2.42,6,109.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
526,3a1983b3,Fiji,20250906,Samoa,23778ceb,aa4c09a1,8.0,1.88,4,80.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
527,3a1983b3,Fiji,20250915,Canada,1039f7c4,55e97718,8.0,0.88,2,79.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
test_match_results = test_matches[['match_id', 'date', 'season', 'venue_id', 'venue', 'home_team',
                                    'home_team_id', 'away_team', 'away_team_id', 'home_score', 'away_score']]

test_match_results['winner'] = np.where(test_match_results['home_score'] > test_match_results['away_score'],
                                        test_match_results['home_team'], test_match_results['away_team'])


test_match_results

In [49]:
match_stats = match_stats[match_stats['team'].isin(test_nations['team_name'])]

In [50]:
## Data validation

match_stats = match_stats.rename(columns = {"team": "team_name"})

match_stats = match_stats[match_stats['turnovers_lost'] < 30]


In [51]:
match_stats = pd.merge(test_nations, match_stats, how = 'inner', on = 'team_name')

match_stats


,team_id,team_name,match_id,carries,line_breaks,tackles_completed,turnovers_lost,turnovers_won,kicks,passes,...,offload,penalty_goals,points,rucks_won,runs,tackles,total_free_kicks_conceded,total_lineouts,turnover_knock_on,turnovers_conceded
0,67d88578,France,3693cfdf,87,6,0,0,0,28,112,...,5,3,35,70,87,172,0,16,4,8
1,67d88578,France,48f165d0,93,5,148,13,2,20,119,...,7,1,17,75,93,148,0,18,8,13
2,67d88578,France,4a49c716,124,7,0,0,0,33,157,...,9,2,37,91,124,100,0,18,7,10
3,67d88578,France,4deaa73c,140,7,117,11,3,23,174,...,11,4,45,103,140,117,1,8,4,11
4,67d88578,France,5222ebe1,96,7,0,0,0,33,131,...,9,3,42,62,96,187,2,12,3,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
286,3a1983b3,Fiji,aa4c09a1,99,3,109,8,6,23,130,...,0,0,0,0,0,0,0,0,0,0
287,3a1983b3,Fiji,b901c3ad,109,6,55,18,4,22,177,...,0,0,0,0,0,0,0,0,0,0
288,3a1983b3,Fiji,c0c4c528,116,6,145,13,8,20,161,...,0,0,0,0,0,0,0,0,0,0
289,3a1983b3,Fiji,cc6f63ba,129,5,91,21,6,24,176,...,0,0,0,0,0,0,0,0,0,0


In [52]:
## Joining data
## Columns of stats not collected using the players dataset
team_stats_copy = team_stats
team_stats = team_stats_copy[['team_id', 'team', 'game_date', 'team_vs', 'team_vs_id', 'match_id', '22m_entries', '22m_conversion',
                        'post_contact_metres', 'ruck_speed_0_3_pct', 'ruck_speed_3_6_pct', 'ruck_speed_6_plus_pct',
                        'possession', 'territory', 'scrums_total', 'mauls_total', 'rucks_total']]

team_stats = team_stats.rename(columns={"team": "team_name"}) # renamed for the purpose of joining

In [ ]:
## Join this with data collected from players data set
## - need to designate the home/away attachment before merging with match results

potential_model_set = pd.merge(team_stats, match_stats, how="left", on = ['team_id', 'team_name', 'match_id'])

potential_model_set

,team_id,team_name,game_date,team_vs,team_vs_id,match_id,22m_entries,22m_conversion,post_contact_metres,ruck_speed_0_3_pct,...,offload,penalty_goals,points,rucks_won,runs,tackles,total_free_kicks_conceded,total_lineouts,turnover_knock_on,turnovers_conceded
0,67d88578,France,20220206,Italy,d0cffe5e,4a49c716,NaN,NaN,NaN,NaN,...,9.0,2.0,37.0,91.0,124.0,100.0,0.0,18.0,7.0,10.0
1,67d88578,France,20220212,Ireland,165b36e5,71ccf2fb,NaN,NaN,NaN,NaN,...,8.0,6.0,30.0,79.0,101.0,119.0,1.0,14.0,7.0,12.0
2,67d88578,France,20220226,Scotland,4409215c,eb863c39,NaN,NaN,NaN,NaN,...,10.0,0.0,36.0,37.0,75.0,153.0,1.0,13.0,6.0,13.0
3,67d88578,France,20220311,Wales,094dae60,6b6585fc,NaN,NaN,NaN,NaN,...,10.0,2.0,13.0,68.0,98.0,131.0,1.0,14.0,9.0,15.0
4,67d88578,France,20220319,England,f60f757d,cc3c0a71,NaN,NaN,NaN,NaN,...,13.0,2.0,25.0,70.0,98.0,130.0,0.0,11.0,8.0,12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
524,3a1983b3,Fiji,20241123,Ireland,165b36e5,8923cd55,17.0,3.06,356.0,65.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
525,3a1983b3,Fiji,20250830,Tonga,be110f80,b901c3ad,12.0,2.42,327.0,63.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
526,3a1983b3,Fiji,20250906,Samoa,23778ceb,aa4c09a1,8.0,1.88,286.0,61.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
527,3a1983b3,Fiji,20250915,Canada,1039f7c4,55e97718,8.0,0.88,216.0,70.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
## Feature Engineering




In [31]:
model_set_copy = potential_model_set


all_stats_present = model_set_copy.dropna()

all_stats_present

,team_id,team_name,game_date,team_vs,team_vs_id,match_id,22m_entries,22m_conversion,post_contact_metres,ruck_speed_0_3_pct,...,offload,penalty_goals,points,rucks_won,runs,tackles,total_free_kicks_conceded,total_lineouts,turnover_knock_on,turnovers_conceded
20,67d88578,France,20240202,Ireland,165b36e5,48f165d0,5.0,2.80,175.0,52.0,...,7.0,1.0,17.0,75.0,93.0,148.0,0.0,18.0,8.0,13.0
21,67d88578,France,20240210,Scotland,4409215c,5adf0a01,9.0,1.44,273.0,58.0,...,4.0,2.0,20.0,76.0,89.0,153.0,1.0,10.0,8.0,15.0
22,67d88578,France,20240225,Italy,d0cffe5e,c850930c,10.0,0.70,360.0,60.0,...,13.0,2.0,13.0,100.0,138.0,118.0,0.0,16.0,11.0,19.0
23,67d88578,France,20240310,Wales,094dae60,4deaa73c,4.0,5.25,260.0,55.0,...,11.0,4.0,45.0,103.0,140.0,117.0,1.0,8.0,4.0,11.0
24,67d88578,France,20240316,England,f60f757d,a9335d7e,6.0,4.67,118.0,38.0,...,10.0,4.0,33.0,91.0,125.0,77.0,1.0,17.0,8.0,15.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
429,d75e45cc,New Zealand,20250816,Argentina,11392c0c,6a36aeb1,8.0,4.75,221.0,49.0,...,4.0,1.0,41.0,80.0,99.0,155.0,2.0,15.0,5.0,13.0
430,d75e45cc,New Zealand,20250823,Argentina,11392c0c,c6b3a680,5.0,3.40,154.0,69.0,...,5.0,2.0,23.0,64.0,83.0,185.0,1.0,14.0,13.0,15.0
431,d75e45cc,New Zealand,20250906,South Africa,9b93d911,1945be6e,5.0,4.20,221.0,30.0,...,5.0,1.0,24.0,75.0,95.0,146.0,0.0,15.0,9.0,14.0
433,d75e45cc,New Zealand,20250927,Australia,b904fef9,da407af5,6.0,4.00,307.0,67.0,...,5.0,3.0,33.0,124.0,162.0,159.0,0.0,15.0,3.0,7.0
